# Week 2: EDA & Model Training
Fraud Detection Engine — exploratory data analysis notebook.

**Dataset:** [Kaggle Credit Card Fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)  
Place `creditcard.csv` in this `notebooks/` folder before running.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

df = pd.read_csv("creditcard.csv")
print(f"Rows: {len(df):,}  Columns: {df.shape[1]}")
df.head()

## 1. Class imbalance — the central problem

In [ ]:
counts = df["Class"].value_counts()
fraud_pct = df["Class"].mean() * 100
print(f"Legit transactions:  {counts[0]:,}")
print(f"Fraud transactions:  {counts[1]:,}")
print(f"Fraud rate:          {fraud_pct:.3f}%")

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(["Legit", "Fraud"], counts.values, color=["steelblue", "tomato"])
ax.set_title("Class distribution (raw)")
ax.set_ylabel("Count")
for i, v in enumerate(counts.values):
    ax.text(i, v + 500, f"{v:,}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()
print("\nThis imbalance is why accuracy is a useless metric here.")
print("A model predicting ALL legit would be 99.83% accurate but catch 0 fraud.")

## 2. Transaction amount distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3))

for ax, label, color in zip(
    axes, ["Legit (0)", "Fraud (1)"], ["steelblue", "tomato"], strict=False
):
    cls = 0 if "Legit" in label else 1
    data = df[df["Class"] == cls]["Amount"]
    ax.hist(data, bins=50, color=color, alpha=0.7, edgecolor="white")
    ax.set_title(f"Amount distribution — {label}")
    ax.set_xlabel("Amount ($)")
    ax.set_ylabel("Count")
    ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"${x:,.0f}"))

plt.tight_layout()
plt.show()

print(
    f"Legit  — mean: ${df[df['Class'] == 0]['Amount'].mean():.2f}  median: ${df[df['Class'] == 0]['Amount'].median():.2f}"
)
print(
    f"Fraud  — mean: ${df[df['Class'] == 1]['Amount'].mean():.2f}  median: ${df[df['Class'] == 1]['Amount'].median():.2f}"
)

## 3. Time of day pattern

In [ ]:
df["hour"] = (df["Time"] % 86400) // 3600

fig, ax = plt.subplots(figsize=(10, 3))
fraud_by_hour = df[df["Class"] == 1].groupby("hour").size()
legit_by_hour = df[df["Class"] == 0].groupby("hour").size()
fraud_rate_by_hour = (fraud_by_hour / (fraud_by_hour + legit_by_hour) * 100).fillna(0)

ax.bar(fraud_rate_by_hour.index, fraud_rate_by_hour.values, color="tomato", alpha=0.8)
ax.set_title("Fraud rate by hour of day")
ax.set_xlabel("Hour (0 = midnight)")
ax.set_ylabel("Fraud rate (%)")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:.2f}%"))
plt.tight_layout()
plt.show()
print("Higher fraud rate at night hours — this becomes an is_night feature.")

## 4. Feature engineering

In [ ]:
def engineer_features(df):
    df = df.copy()
    df["amount_log"] = np.log1p(df["Amount"])  # normalise skewed amount
    df["hour_of_day"] = (df["Time"] % 86400) // 3600
    df["is_night"] = df["hour_of_day"].apply(lambda h: 1 if h >= 23 or h <= 5 else 0)
    df = df.drop(columns=["Time", "Amount"])
    return df


df_feat = engineer_features(df)
feature_cols = [c for c in df_feat.columns if c != "Class"]
print(f"Feature count: {len(feature_cols)}")
print(f"Features: {feature_cols}")
df_feat.head()

## 5. Train / test split + SMOTE

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X = df_feat[feature_cols]
y = df_feat["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print(f"Train before SMOTE: {y_train.sum():,} fraud / {len(y_train):,} total")
print(f"Train after  SMOTE: {y_train_res.sum():,} fraud / {len(y_train_res):,} total")
print(f"Test (untouched):   {y_test.sum():,} fraud / {len(y_test):,} total")

## 6. Train XGBoost

In [ ]:
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    eval_metric="logloss",
    verbosity=0,
)
model.fit(X_train_res, y_train_res)
print("Training complete.")

## 7. Threshold tuning — the most important step

In [ ]:
from sklearn.metrics import classification_report, precision_recall_curve

y_proba = model.predict_proba(X_test_scaled)[:, 1]
precision, recall, thresholds = precision_recall_curve(y_test, y_proba)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(thresholds, precision[:-1], label="Precision", color="steelblue")
ax.plot(thresholds, recall[:-1], label="Recall", color="tomato")
ax.axvline(x=0.4, color="gray", linestyle="--", label="Threshold = 0.4")
ax.set_xlabel("Threshold")
ax.set_ylabel("Score")
ax.set_title("Precision vs Recall at different thresholds")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

THRESHOLD = 0.4
y_pred = (y_proba >= THRESHOLD).astype(int)
print(f"Results at threshold = {THRESHOLD}:")
print(classification_report(y_test, y_pred, target_names=["Legit", "Fraud"]))

## 8. Save model

In [ ]:
import os

import joblib

os.makedirs("../models", exist_ok=True)
joblib.dump(model, "../models/xgb_fraud_v1.joblib")
joblib.dump(scaler, "../models/scaler_v1.joblib")

with open("../models/feature_columns.txt", "w") as f:
    f.write("\n".join(feature_cols))

print("Saved:")
print("  models/xgb_fraud_v1.joblib")
print("  models/scaler_v1.joblib")
print("  models/feature_columns.txt")
print("\nReady for week 3 — scoring API.")